# Phase 1: Exploratory Data Analysis (EDA) - Loan Default Prediction

**Objective:** Uncover underlying patterns, anomalies, missingness, and structural relationships in the loan portfolio to inform data preprocessing (Phase 2), feature engineering (Phase 3), and modeling (Phase 4).

> **Data Leakage Rule:** We exclusively use `data/train.csv` during exploration and model training. Never touch `data/test.csv` during this phase.

## 1. Environment Setup & Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display formatting
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment successfully configured.")

In [ ]:
TRAIN_PATH = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
print(f"Loading training set from: {TRAIN_PATH.resolve()}")

df_train = pd.read_csv(TRAIN_PATH)
print(f"Dimensions: {df_train.shape[0]:,} rows x {df_train.shape[1]} columns")
df_train.head()

## 2. Target Variable & Class Balance Analysis

Evaluating baseline default rate. In loan underwriting, false negatives (approving a borrower who defaults) carry high financial loss compared to false positives (turning away a creditworthy borrower).

In [ ]:
target_counts = df_train['loan_default'].value_counts()
target_pct = df_train['loan_default'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct
})
print("Target Class Distribution:")
display(summary_target)

# Plot target class balance
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Non-Default (0)', 'Default (1)'], target_counts.values, color=['#2b5c8f', '#d95f02'], width=0.45)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:,}\n({height/len(df_train):.1%})',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Number of Loans')
ax.set_title('Loan Default Target Balance (Class Distribution)', fontsize=12, pad=12)
ax.set_ylim(0, target_counts.max() * 1.18)
plt.tight_layout()
plt.show()

## 3. Missing Value Audit (Explicit & Implicit)

We inspect:
1. **Explicit NaN:** Missing fields in customer applications.
2. **Implicit / Disguised Missingness:** When a bureau score is `0` indicating no credit history (New-to-Credit).

In [ ]:
# 1. Explicit Missingness
missing_counts = df_train.isnull().sum()
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts[missing_counts > 0],
    'Missing_Percentage (%)': (missing_counts[missing_counts > 0] / len(df_train)) * 100
})
print("Explicit Missing Values (NaN):")
display(missing_df if not missing_df.empty else "No explicit NaN values found.")

# 2. Implicit Missingness: Bureau Score = 0
ntc_count = (df_train['PERFORM_CNS.SCORE'] == 0).sum()
print(f"\nImplicit Missingness (New to Credit / No Bureau Score):")
print(f"  - Borrowers with 0 Bureau Score: {ntc_count:,} ({ntc_count/len(df_train):.2%})")

## 4. Feature Taxonomy & Domain Categorization

We map the 41 raw features into 9 distinct credit risk domains.

In [ ]:
feature_groups = {
    'Identifiers': ['UniqueID', 'branch_id', 'supplier_id', 'manufacturer_id', 'Current_pincode_ID', 'State_ID', 'Employee_code_ID'],
    'Loan & Asset': ['disbursed_amount', 'asset_cost', 'ltv'],
    'Borrower Demographics': ['Date.of.Birth', 'Employment.Type'],
    'Loan Disbursal': ['DisbursalDate'],
    'KYC Identity Flags': ['MobileNo_Avl_Flag', 'Aadhar_flag', 'PAN_flag', 'VoterID_flag', 'Driving_flag', 'Passport_flag'],
    'Bureau Score': ['PERFORM_CNS.SCORE', 'PERFORM_CNS.SCORE.DESCRIPTION'],
    'Primary Credit Profile': [col for col in df_train.columns if col.startswith('PRI.')] + ['PRIMARY.INSTAL.AMT'],
    'Secondary Credit Profile': [col for col in df_train.columns if col.startswith('SEC.')] + ['SEC.INSTAL.AMT'],
    'Credit Bureau Behavioral History': ['NEW.ACCTS.IN.LAST.SIX.MONTHS', 'DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS', 'AVERAGE.ACCT.AGE', 'CREDIT.HISTORY.LENGTH', 'NO.OF_INQUIRIES'],
    'Target': ['loan_default']
}

for category, cols in feature_groups.items():
    print(f"{category:35s} : {len(cols):2d} features")

total_mapped = sum(len(cols) for cols in feature_groups.values())
print(f"\nTotal mapped features: {total_mapped} of {df_train.shape[1]}")

## 5. Financial Risk Drivers (LTV, Disbursed Amount, Asset Cost)

Loan-to-Value (LTV) measures the leverage of the borrower. High LTV implies minimal borrower equity/down payment.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Disbursed Amount Distribution by Default Status
sns.boxplot(x='loan_default', y='disbursed_amount', data=df_train, ax=axes[0], palette=['#2b5c8f', '#d95f02'], showfliers=False)
axes[0].set_title('Disbursed Amount vs Default')
axes[0].set_xticklabels(['Non-Default (0)', 'Default (1)'])

# 2. LTV Distribution by Default Status
sns.boxplot(x='loan_default', y='ltv', data=df_train, ax=axes[1], palette=['#2b5c8f', '#d95f02'], showfliers=False)
axes[1].set_title('LTV (%) vs Default')
axes[1].set_xticklabels(['Non-Default (0)', 'Default (1)'])

# 3. LTV Quintiles Default Rate
df_train['ltv_quintile'] = pd.qcut(df_train['ltv'], q=5, duplicates='drop')
ltv_default_rates = df_train.groupby('ltv_quintile', observed=False)['loan_default'].mean() * 100
ltv_default_rates.plot(kind='bar', ax=axes[2], color='#d95f02', edgecolor='black', alpha=0.85)
axes[2].set_title('Default Rate by LTV Quintile (%)')
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_xlabel('LTV Range')
axes[2].set_xticklabels([f"{interval.left:.0f}-{interval.right:.0f}%" for interval in ltv_default_rates.index], rotation=0)

for p in axes[2].patches:
    axes[2].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3),
                     textcoords='offset points')

plt.tight_layout()
plt.show()

## 6. Credit Bureau Scores & Delinquency Behavior

Examining CRISIL/HighMark credit scores (`PERFORM_CNS.SCORE`), bureau risk descriptions, and inquiry frequency (`NO.OF_INQUIRIES`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Bureau Score Distribution
sns.histplot(df_train['PERFORM_CNS.SCORE'], bins=40, kde=False, color='#2b5c8f', ax=axes[0])
axes[0].set_title('Bureau Credit Score Distribution (Notice the 0 spike for NTC)')
axes[0].set_xlabel('Score (0 = No Bureau History)')
axes[0].set_ylabel('Borrower Count')

# 2. Inquiries vs Default Rate
df_train['inquiry_bucket'] = df_train['NO.OF_INQUIRIES'].apply(lambda x: str(x) if x < 4 else '4+')
inq_default = df_train.groupby('inquiry_bucket', observed=False)['loan_default'].agg(['count', 'mean'])
inq_default['mean_pct'] = inq_default['mean'] * 100

bars = axes[1].bar(inq_default.index, inq_default['mean_pct'], color='#7570b3', edgecolor='black', width=0.5)
axes[1].set_title('Default Rate by Number of Credit Inquiries (%)')
axes[1].set_xlabel('Number of Inquiries')
axes[1].set_ylabel('Default Rate (%)')

for bar in bars:
    height = bar.get_height()
    axes[1].annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4), textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Bureau Risk Categories vs Default Rate
risk_summary = df_train.groupby('PERFORM_CNS.SCORE.DESCRIPTION')['loan_default'].agg(
    Count='count',
    Default_Rate=lambda x: x.mean() * 100
).sort_values('Default_Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
risk_summary['Default_Rate'].plot(kind='barh', color='#d95f02', edgecolor='black', ax=ax)
ax.set_title('Default Rate by Bureau Risk Rating (%)', fontsize=12, pad=12)
ax.set_xlabel('Default Rate (%)')
ax.set_ylabel('Bureau Rating')
ax.axvline(df_train['loan_default'].mean() * 100, color='black', linestyle='--', label=f'Portfolio Average ({df_train["loan_default"].mean()*100:.1f}%)')
ax.legend()

plt.tight_layout()
plt.show()

display(risk_summary)

## 7. Borrower Demographics & Employment Analysis

Comparing risk across `Employment.Type` (`Salaried`, `Self employed`, and missing).

In [ ]:
emp_series = df_train['Employment.Type'].fillna('Missing')
emp_summary = df_train.groupby(emp_series)['loan_default'].agg(
    Count='count',
    Defaults='sum',
    Default_Rate=lambda x: x.mean() * 100
)
print("Employment Type Risk Breakdown:")
display(emp_summary)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(emp_summary.index, emp_summary['Default_Rate'], color=['#7570b3', '#2b5c8f', '#e7298a'], width=0.45)
ax.set_title('Default Rate by Employment Type (%)')
ax.set_ylabel('Default Rate (%)')
ax.axvline(0, color='grey', linewidth=0.8)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Correlation Heatmap of Primary Numerical Features

In [ ]:
key_num_cols = [
    'disbursed_amount', 'asset_cost', 'ltv', 'PERFORM_CNS.SCORE',
    'PRI.NO.OF.ACCTS', 'PRI.ACTIVE.ACCTS', 'PRI.OVERDUE.ACCTS',
    'PRI.CURRENT.BALANCE', 'PRI.SANCTIONED.AMOUNT', 'PRIMARY.INSTAL.AMT',
    'NEW.ACCTS.IN.LAST.SIX.MONTHS', 'DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS',
    'NO.OF_INQUIRIES', 'loan_default'
]

corr_matrix = df_train[key_num_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', cbar=True, square=True, linewidths=0.5)
plt.title('Correlation Matrix of Key Numerical & Credit Features', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

print("Correlations with Target (loan_default):")
display(corr_matrix['loan_default'].sort_values(ascending=False))

## 9. Key Findings & Strategic Hypotheses for Phase 2 & 3

| Finding | Observation | Preprocessing / Feature Engineering Action |
| :--- | :--- | :--- |
| **Class Imbalance** | 21.7% default rate (approx. 3.6:1 ratio). | Accuracy is inappropriate. Target ROC-AUC, PR-AUC, and tune decision thresholds in Phase 5. |
| **Loan-to-Value (LTV)** | Default jumps from 15.0% (<66% LTV) to 26.6% (>85% LTV). | Key feature. Check for outliers (>100% LTV) in Phase 2. Create LTV risk-tier features in Phase 3. |
| **Implicit Missingness** | Over 50% of applicants have 0 bureau score ("No Bureau History Available"). | Create a binary `is_new_to_credit` flag in Phase 3 so models distinguish thin-file borrowers from bad-score borrowers. |
| **Credit Inquiries** | Applicants with 4+ inquiries default at 34.9% vs 21.1% for 0 inquiries. | Strong indicator of credit distress/hunger. Retain as a critical numeric signal. |
| **Employment Type** | Self-employed applicants default at 22.8% vs 20.3% for salaried (missing ~3.3%). | Impute missing as a separate `'Missing'` or `'Unknown'` category in Phase 2. |
| **Account Age Durations** | `AVERAGE.ACCT.AGE` & `CREDIT.HISTORY.LENGTH` are strings (e.g. `'1yrs 11mon'`). | Must parse into numeric months in Phase 3. |
| **Date Formats** | `Date.of.Birth` and `DisbursalDate` are DD-MM-YY strings. | Compute applicant age at disbursal in Phase 3. |